<a href="https://colab.research.google.com/github/rlaj0730-design/deep-learning-study/blob/main/ex04_%EC%82%AC%EB%9E%8C_%EC%82%AC%EC%A7%84_%EB%B6%84%EB%A5%98_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 실습.학습목표
- 앞서 배운 내용을 활용해 3가지 인물을 분류하는 모델을 직접 만들어보자
  - 조이, 카리나, 태연
- 직접 CNN 레이어 만들기, InceptionV3나 VGG16같은 사전학습 모델 활용하기 등 자유롭게 선택해서 모델을 구성해보세요

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/딥러닝 기초"

/content/drive/MyDrive/Colab Notebooks/딥러닝 기초


In [ ]:
!ls "./data/person_train/"

조이  카리나  태연


In [ ]:
import numpy as np

from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
face_data = image_dataset_from_directory(
    "./data/person_train",
    image_size=(224, 224),
    batch_size=32
)

print(face_data.class_names)

Found 1498 files belonging to 3 classes.
['조이', '카리나', '태연']


In [ ]:
X_train = []
y_train = []

for images, labels in face_data.as_numpy_iterator():
    X_train.append(images)
    y_train.append(labels)

X_train = np.concatenate(X_train)
y_train = np.concatenate(y_train)

print(X_train.shape, y_train.shape)

(1498, 224, 224, 3) (1498,)


In [ ]:
X_train_face, X_test_face, y_train_face, y_test_face = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print("훈련용 데이터:", X_train_face.shape, y_train_face.shape)
print("평가용 데이터:", X_test_face.shape, y_test_face.shape)

훈련용 데이터: (1198, 224, 224, 3) (1198,)
평가용 데이터: (300, 224, 224, 3) (300,)


In [ ]:
face_vgg16 = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

In [ ]:
face_model = Sequential()

face_model.add(face_vgg16)
face_model.add(Flatten())
face_model.add(Dense(units=128, activation="relu"))
face_model.add(Dense(units=64, activation="relu"))
face_model.add(Dense(units=3, activation="softmax"))

In [ ]:
face_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=Adam(learning_rate=0.0001),
    metrics=["accuracy"]
)

In [ ]:
h_face = face_model.fit(
    X_train_face,
    y_train_face,
    epochs=10
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 131s 2s/step - accuracy: 0.3539 - loss: 1.4509
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 458ms/step - accuracy: 0.4574 - loss: 1.0678
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 464ms/step - accuracy: 0.5100 - loss: 0.9827
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 472ms/step - accuracy: 0.6060 - loss: 0.8530
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 468ms/step - accuracy: 0.6853 - loss: 0.7440
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 465ms/step - accuracy: 0.7446 - loss: 0.6415
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 465ms/step - accuracy: 0.8030 - loss: 0.4853
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 466ms/step - accuracy: 0.7855 - loss: 0.4949
Epoch 9/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 466ms/step - accuracy: 0.8965 - loss: 0.3011
Epoch 10/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 465ms/step - accuracy: 0.9157 - loss: 0.2434


In [ ]:
predict_result = face_model.predict(X_test_face)
y_pred = predict_result.argmax(1)

print("정답 번호 순서:", face_data.class_names)
print(classification_report(y_test_face, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 232ms/step
정답 번호 순서: ['조이', '카리나', '태연']
              precision    recall  f1-score   support

           0       0.87      0.78      0.82        99
           1       0.78      0.71      0.75       107
           2       0.74      0.89      0.81        94

    accuracy                           0.79       300
   macro avg       0.80      0.79      0.79       300
weighted avg       0.80      0.79      0.79       300

